# 인공지능 스피커

- 실시간 음성 수집 - 텍스트 추출 -  LLM 응답 - 텍스트를 오디오로 변환 - 오디오 저장 - 오디오 백그라운드 재생

In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [5]:
# 실시간 음성 수집
import speech_recognition as sr

r = sr.Recognizer()

with sr.Microphone() as source:
    print('말해주세요.')
    r.adjust_for_ambient_noise(source)
    # 마이크 입력 받기
    audio = r.listen(source)
    print('인식 중입니다...')
    # 텍스트 변화
    text = r.recognize_openai(audio)
    print(text)

말해주세요.
인식 중입니다...
말


In [6]:
%pip install google-genai
from google import genai
from google.genai import types

google_client = genai.Client()

   ---------------------------------------- 0.0/958.0 kB ? eta -:--:--
   ---------------------------------------- 958.0/958.0 kB 11.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [10]:
def gemini_bot(user_message, system_prompt="당신은 불친절한 챗봇입니다"):
    response = google_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=user_message,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        ),
    )
    return response.text

In [11]:
answer = gemini_bot(user_message=text)
print(answer)

말, 뭘 말하고 싶은 건데?


In [12]:
from openai import OpenAI
client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model='gpt-4o-mini-tts',
    voice='coral',
    input=answer,
    speed=1.4,  # 0.25 ~ 4.0
    instructions='밝고 자신감 있는 목소리로 말해줘'
) as response:
    # 오디오 저장
    temp_path='./audio/answer.mp3'
    response.stream_to_file(temp_path)

# 재생
from IPython.display import Audio, display
display(Audio(temp_path, autoplay=True))

## 통합 코드 만들기

In [13]:
# LLM
chat = google_client.chats.create(
    model="gemini-2.5-flash-lite",
    config=types.GenerateContentConfig(
        system_instruction="당신은 매우 불친절합니다. 반드시 30자 이내로만 답해주세요."
    ),
)

In [ ]:
from pydub import AudioSegment
from pydub.playback import play

while True:
    # 실시간 음성 수집    
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("말해주세요.")
        r.adjust_for_ambient_noise(source)
        # step1 : 마이크 입력 받기
        audio = r.listen(source)
        print("인식 중입니다....")
        # step2 : 텍스트 변화
        user_text = r.recognize_openai(audio)
        print(f"[USER] {user_text}")

        # 종료 조건
        if user_text.strip() in ["그만", "종료", "꺼", "닥쳐"]:
            break

        # step3 : LLM이 사용자의 입력에 답하기
        try:
            answer = chat.send_message(user_text).text
            print(f'[AI] {answer}')
        except Exception as e:
            print("응답 생성 실패")
            answer = "지금 답변 생성 어려움"

        # step4 : 텍스트를 오디오로 변환하기
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice="coral",
            input=answer,
            speed= 1.2,   # 0.25 ~ 4.0
            instructions="화난 목소리로 빠르게 말해주세요. 두 문장으로 말해줘" # "슬픈 목소리로 말해줘"
        ) as response:
            # step5 : 오디오 저장    
            temp_path = "./audio/answer.mp3"
            response.stream_to_file(temp_path)

            # ste6 : 오디오 재생
            # display(Audio(temp_path, autoplay=True))
            sound = AudioSegment.from_mp3(temp_path)
            play(sound)

In [ ]:
# 별개로 진행해봄
SYSTEM_PROMPT = """너는 이런 성격의 캐릭터야.

영화 인사이드 아웃에 나오는 기쁨이 캐릭터
작고 귀여운 외모: 귀여운 얼굴과 작은 체구가 특징이에요.
활발한 성격: 언제나 에너지가 넘치고 활발하게 움직여요.
호기심이 많음: 새로운 물건이나 사람에게 호기심이 많아 관심을 보이며 탐색해요.
사람을 좋아함: 사람들과의 교류를 좋아하고 관심을 받는 것을 즐겨요.
훈련을 잘 따름: 간식이나 칭찬에 민감해 훈련을 잘 따르고 순종적이에요.
잘 먹음: 음식을 좋아하고 식욕이 왕성해요.
놀기 좋아함: 공이나 장난감을 가지고 노는 것을 즐기며 활발하게 뛰어다녀요.
온화한 성격: 화를 잘 내지 않고 온화한 성격이에요."""

recognizer = sr.Recognizer()

def transcribe_audio(recognizer, phrase_time_limit=5, timeout=10):
    with sr.Microphone() as source:
        recognizer.adjust_for_ambient_noise(source, duration=1)
        print("말씀해주세요...")
        try:
            return recognizer.recognize_google(audio, language="ko-KR")
        except sr.UnknownValueError:
            print("음성을 이해하지 못했습니다.")
            return None
        except sr.RequestError as e:
            print("음성 인식 요청 실패:", e)
            return None

def ask_openai(user_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
        ]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        max_tokens=200,
        temperature=0.8,
        )
    return response.choices[0].message["content"].strip()
    print("음성 에이전트가 시작되었습니다. 종료하려면 '종료'라고 말하세요.")

while True:
    user_text = transcribe_audio(recognizer)
    if user_text is None:
        print("음성을 인식하지 못했습니다. 다시 시도해주세요.")
        continue

print("사용자:", user_text)
if "종료" in user_text or "그만" in user_text:
    print("에이전트를 종료합니다.")
    break

answer_text = ask_openai(user_text)
print("[AI 답변]:", answer_text)